In [ ]:
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns

import boto3
import json
import yaml
from pathlib import Path
from io import BytesIO

In [ ]:
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

In [ ]:
with open("config/storage.yaml", "r") as f:
    config = yaml.safe_load(f)

In [ ]:
AWS_PROFILE = config["aws"]["profile"]
AWS_REGION = config["aws"]["region"]
BUCKET = config["aws"]["bucket"]
RAW_PREFIX = config["s3"]["prefixes"]["raw"]

session = boto3.Session(
    profile_name=AWS_PROFILE,
    region_name=AWS_REGION,
)

s3 = session.client("s3")

print(f"Bucket: {BUCKET}")
print(f"Raw prefix: {RAW_PREFIX}")

In [ ]:
objects = []

paginator = s3.get_paginator("list_objects_v2")

for page in paginator.paginate(
    Bucket=BUCKET,
    Prefix=RAW_PREFIX,
):
    for obj in page.get("Contents", []):
        objects.append({
            "key": obj["Key"],
            "size_mb": obj["Size"] / 1024**2,
            "last_modified": obj["LastModified"],
        })

inventory = pd.DataFrame(objects)

inventory.sort_values("key").head(20)

In [ ]:
def load_s3_dataset(key: str) -> pd.DataFrame:
    response = s3.get_object(
        Bucket=BUCKET,
        Key=key,
    )

    body = BytesIO(response["Body"].read())

    if key.lower().endswith(".csv"):
        return pd.read_csv(body)

    if key.lower().endswith((".parquet", ".pq")):
        return pd.read_parquet(body)

    raise ValueError(f"Unsupported file format: {key}")